# 12 · Claude Code and computer use

**Goal:** understand two agents Anthropic built, Claude Code and computer use, well enough to use the first and to know what the second needs before you try it.

| | |
|---|---|
| **You will learn** | What Claude Code is and how to work with it, how to plug in your own MCP server, how computer use works, what it costs in tokens, and how to run it safely |
| **Prerequisites** | [Notebook 05](05_tool_use.ipynb) (the tool loop). [Notebook 10](10_model_context_protocol.ipynb) for section 2 |
| **Run time** | Under a minute |
| **Cost** | **Free.** This notebook makes no paid calls. Section 3 uses the free token-counting endpoint |
| **Paid cells** | None |

**This notebook is mostly a guide.** Claude Code is a separate app that runs in your terminal, and computer use needs a sandboxed virtual machine, so neither belongs inside a notebook. The code cells here only check your setup and count tokens, which costs nothing.

## Why these two?

Both are **agents**, built by Anthropic with the same pieces you've used in this repo:

| Piece | Where you built it | Claude Code | Computer use |
|---|---|---|---|
| Tools Claude can request | Notebook 05 | Read, edit, search files, run shell commands | Screenshot, click, type, scroll |
| A loop that runs until the task is done | Notebook 05 | Yes | Yes |
| Seeing the result of each action | Notebook 11 | Reads files before editing, runs tests | A screenshot after each action |
| Extra tools from outside | Notebook 10 | MCP servers | No |

Studying them shows what a working agent looks like.

## Setup

Only the client is needed here, for the free token-counting call in section 3. There's no budget cell, because nothing in this notebook is billed.

In [1]:
import os
import shutil
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

client = Anthropic()  # finds ANTHROPIC_API_KEY by itself
MODEL = "claude-sonnet-5-5"

print("Ready. Using model:", MODEL)

Ready. Using model: claude-sonnet-5-5


## 1. Claude Code: an agent in your terminal

Claude Code is a coding assistant that runs in your terminal. You describe a task in plain English, and Claude reads your files, edits them, runs commands such as your tests, and checks the results, asking your permission along the way.

It's the tool loop from notebook 05, with tools for working on a codebase:

- **Files:** search, read and edit files in your project.
- **Terminal:** run shell commands, such as tests, builds and git.
- **Web:** search and fetch documentation.
- **MCP:** connect extra tools from MCP servers, including your own (section 2).

It also runs in VS Code, JetBrains IDEs, a desktop app and on the web. This guide covers the terminal version.

### Install it

Run one of these in a terminal. They come from the [Claude Code quickstart](https://code.claude.com/docs/en/quickstart), which is the place to check if anything has changed.

| System | Command |
|---|---|
| macOS, Linux, WSL (recommended) | `curl -fsSL https://claude.ai/install.sh \| bash` |
| Windows PowerShell | `irm https://claude.ai/install.ps1 \| iex` |
| macOS with Homebrew | `brew install --cask claude-code` |
| Windows with WinGet | `winget install Anthropic.ClaudeCode` |

Then open a **new** terminal and check it worked:

```bash
claude --version
```

### Start it and sign in

Go to a project folder and run `claude`:

```bash
cd your-project
claude
```

The first time, it asks you to sign in. You can use a Claude subscription (Pro, Max, Team or Enterprise) or a Claude Console account, which bills your API credits.

> **Watch your spending.** If you sign in with a Console account, Claude Code's usage is billed to your API account, and the `track()` function in these notebooks can't see it. Claude Code gets its own "Claude Code" workspace in the Console, so you can follow its costs there. Keep a spend limit set on the **Billing** page.

### The workflow that works best

**1. Give it a memory with `CLAUDE.md`.** Inside Claude Code, run `/init`. Claude scans your project and writes a `CLAUDE.md` file with build commands, test instructions and conventions. That file is loaded into **every** session, so Claude doesn't have to rediscover your project each time. If a `CLAUDE.md` already exists, `/init` suggests improvements instead of overwriting it.

| File | Who it's for |
|---|---|
| `./CLAUDE.md` or `./.claude/CLAUDE.md` | Everyone on the project. Commit it |
| `./CLAUDE.local.md` | Just you, in this project. Add it to `.gitignore` |
| `~/.claude/CLAUDE.md` | Just you, in every project |

Type `/memory` in a session to see and open these files. To add a rule, you can also just ask: "Add to CLAUDE.md that tests must pass before committing."

> **Different from the course:** the course adds notes with a `#` shortcut. The current Claude Code docs don't mention it. Use `/memory` or ask Claude directly instead.

**2. Context, then a plan, then code.** This is the course's key workflow:

1. **Context:** "Read `tools.py` and `server.py`." Point Claude at the files that matter.
2. **Plan, with no code yet:** "Plan how to add a search tool. Don't write any code yet."
3. **Implement:** "Implement the plan."

Claude Code has a **plan mode** built for step 2. In it, Claude reads and explores but doesn't edit your files until you approve the plan. Press `Shift+Tab` to switch modes during a session, or start in plan mode with:

```bash
claude --permission-mode plan
```

**For more reliable code, add tests first:** give the context, ask Claude to list test cases, have it write the tests, then ask for code that makes them pass. Clear tests give Claude a target it can check by itself.

### Permission modes

Claude Code asks before acting, and the **permission mode** decides how often:

| Mode | What Claude does without asking |
|---|---|
| `default` (shown as Manual) | Reads files. Asks before edits and most shell commands |
| `acceptEdits` | Also edits files and runs simple file commands such as `mkdir` |
| `plan` | Reads and explores only. No edits to your source until you approve a plan |
| `auto` | Runs without routine prompts, while a background check reviews risky actions |

There are also `dontAsk` and `bypassPermissions`. **Avoid `bypassPermissions` outside a throwaway sandbox**, because it skips the prompts that protect you.

### Commands you'll use daily

| Where | Command | What it does |
|---|---|---|
| Terminal | `claude` | Start a session |
| Terminal | `claude "fix the failing test"` | Start a session with a first request |
| Terminal | `claude -p "explain this function"` | Answer once and exit |
| Terminal | `claude -c` | Continue your last conversation in this folder |
| In a session | `/init` | Create or improve `CLAUDE.md` |
| In a session | `/clear` | Clear the conversation and start fresh |
| In a session | `/help` | List the available commands |

## 2. Plugging your own MCP server into Claude Code (free)

Claude Code **is an MCP client**, the same role your `MCPClient` class played in notebook 10. So the Halvora document server you built there can plug straight into Claude Code, with no code changes.

You register a server once with `claude mcp add`. The `--` separates Claude Code's own options from the command that starts your server:

```bash
claude mcp add [options] <name> -- <command> [args...]
```

The cell below checks you have what you need. It prints only yes or no, so no file paths from your machine end up saved in this notebook.

In [2]:
repo = Path.cwd().parent   # this notebook runs from the notebooks/ folder

checks = {
    "Claude Code installed (the `claude` command)": shutil.which("claude") is not None,
    "This repo's virtual environment (.venv)": (repo / ".venv").exists(),
    "The Halvora MCP server from notebook 10": Path("halvora_mcp/server.py").exists(),
}

for label, ok in checks.items():
    print(f"{'yes' if ok else 'NO ':<5}{label}")

if not checks["The Halvora MCP server from notebook 10"]:
    print("\nRun notebook 10 first: it builds the server this section connects to.")

yes  Claude Code installed (the `claude` command)
yes  This repo's virtual environment (.venv)
yes  The Halvora MCP server from notebook 10


**Register the server.** Run this from the **repo root** (the folder that holds `.venv`), on macOS, Linux or WSL:

```bash
claude mcp add --transport stdio halvora-docs -- "$(pwd)/.venv/bin/python" "$(pwd)/notebooks/halvora_mcp/server.py"
```

- `halvora-docs` is the name you'll see inside Claude Code.
- `"$(pwd)/..."` makes your shell fill in the full path when you run the command, so the server starts correctly whichever folder Claude Code launches from.
- Using the repo's own `.venv` Python means the server finds the `mcp` package it needs.
- On Windows, the Python path is `.venv\Scripts\python.exe` instead.

**Choose who can use it** with `--scope`:

| Scope | Who can use the server | Where it's stored |
|---|---|---|
| `local` (default) | Only you, in this project | Your user settings |
| `project` | Everyone who clones the project | A `.mcp.json` file in the repo |
| `user` | Only you, in every project | Your user settings |

**Check and manage it:**

| Command | What it does |
|---|---|
| `claude mcp list` | List your servers |
| `claude mcp get halvora-docs` | Show one server's details |
| `claude mcp remove halvora-docs` | Remove it |
| `/mcp` (in a session) | Show which servers are connected |

**Try it.** Start `claude` in the repo and ask:

- "What tools does the halvora-docs server provide?"
- "Using the halvora-docs server, summarize what the cybersecurity document says about the Q4 incident."

Claude Code calls your server's tools through MCP, the same flow you built by hand in notebook 10. Claude Code just plays the client.

## 3. Computer use: an agent that sees a screen

Computer use lets Claude operate a desktop: it looks at a screenshot, decides what to do, and asks your program to click, type or scroll. Then it looks at the next screenshot to see what happened.

```
 1. You send:     the task, plus a screenshot of the desktop
 2. Claude sends: tool_use requests, e.g. left_click at (512, 300)
 3. Your program performs the actions on a virtual machine
 4. You send:     tool_results, including a new screenshot
 5. Repeat until Claude says the task is done
```

It's the tool loop from notebook 05, but the "tools" are a mouse, a keyboard and a camera pointed at the screen. **Anthropic doesn't provide the computer.** You run the virtual machine, perform the actions and take the screenshots, just as you ran your own functions in notebook 05.

**On Sonnet 5.5, computer use is a toolset.** One entry in `tools` gives Claude 17 member tools:

```python
tools = [{"type": "computer_toolset_20260801"}]
```

- It has **no `name`** and **no screen size**. The members include `screenshot`, `zoom`, `left_click`, `double_click`, `left_click_drag`, `scroll`, `type`, `key` and `wait`.
- An optional `configs` map switches members off, for example `{"zoom": {"enabled": False}}`.

> **Different from the course:** older guides declare `{"type": "computer_20251124", "name": "computer", "display_width_px": ..., ...}` with a beta header, or an earlier version of the tool. On the Claude API, Sonnet 5.5 rejects those with a 400 error and accepts only the toolset.

### What the toolset costs before Claude does anything (free)

Tool definitions are sent with every request, as you saw in notebook 05. The computer toolset is large. The token-counting endpoint is free and generates nothing, so we can measure it:

In [3]:
question = [{"role": "user", "content": "Open the settings app and turn on dark mode."}]
toolset = {"type": "computer_toolset_20260801"}
toolset_without_zoom = {"type": "computer_toolset_20260801", "configs": {"zoom": {"enabled": False}}}

without_tools = client.messages.count_tokens(model=MODEL, messages=question).input_tokens
with_toolset = client.messages.count_tokens(model=MODEL, messages=question, tools=[toolset]).input_tokens
without_zoom = client.messages.count_tokens(model=MODEL, messages=question, tools=[toolset_without_zoom]).input_tokens

print(f"No tools               : {without_tools:>6,} input tokens")
print(f"Computer toolset       : {with_toolset:>6,} input tokens  (+{with_toolset - without_tools:,})")
print(f"Toolset, zoom disabled : {without_zoom:>6,} input tokens  (+{without_zoom - without_tools:,})")

extra_cost = (with_toolset - without_tools) * 2.00 / 1_000_000   # Sonnet 5.5 input price
print(f"\nThe toolset alone adds about ${extra_cost:.4f} to EVERY request of the loop.")

assert with_toolset > without_tools
assert without_zoom < with_toolset

No tools               :     23 input tokens
Computer toolset       :  4,545 input tokens  (+4,522)
Toolset, zoom disabled :  4,134 input tokens  (+4,111)

The toolset alone adds about $0.0090 to EVERY request of the loop.


**What to notice**

- Anthropic's pricing page puts the toolset at about 4,500 input tokens, and disabling `zoom` saves about 410. Your numbers should be close.
- That's paid on **every turn**, and each turn also sends a screenshot as an image. A task that takes 30 clicks resends the toolset 30 times, plus 30 screenshots. Computer use is far more expensive per task than the notebooks in this repo.
- **Smaller screenshots are cheaper.** Anthropic suggests around 1024×768 or 1280×720 for desktop tasks. Larger images aren't shrunk for you: above the size limit, they're rejected.

### What the loop looks like

The sketch below shows the shape of a computer use loop. **It is deliberately not a code cell, and you shouldn't run it.** It needs a virtual machine that the `take_screenshot()` and `perform()` placeholders would control.

```python
# A SKETCH, NOT RUNNABLE: take_screenshot() and perform() stand for your own VM code.
messages = [{"role": "user", "content": "Open the settings app and turn on dark mode."}]

for turn in range(20):                       # always cap the number of turns
    response = client.messages.create(
        model=MODEL,
        max_tokens=4000,
        output_config={"effort": "low"},
        tools=[{"type": "computer_toolset_20260801"}],
        messages=messages,
    )
    messages.append({"role": "assistant", "content": response.content})
    if response.stop_reason != "tool_use":
        break                                # Claude says it's done, or it stopped

    results = []
    for block in response.content:           # often several actions per turn
        if block.type != "tool_use":
            continue
        # The ACTION is the block's name, e.g. "left_click", and its input holds the coordinates.
        if block.name in ("screenshot", "zoom"):
            content = [{"type": "image", "source": {
                "type": "base64", "media_type": "image/png", "data": take_screenshot(block.input)}}]
        else:
            perform(block.name, block.input)
            content = "OK"
        results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "toolset_name": "computer",       # every result must echo this, or it's rejected
            "content": content,
        })
    messages.append({"role": "user", "content": results})
```

Compared with notebook 05:

- The action is the tool_use block's **`name`**, such as `left_click`, and each block carries `toolset_name: "computer"`.
- Every `tool_result` must **echo `toolset_name: "computer"`**.
- Only `screenshot` and `zoom` results need an image. For the other actions, `"OK"` is enough.

### Running it safely

Computer use acts on a real screen, so mistakes have real effects. Anthropic's guidance, in short:

1. **Use a dedicated virtual machine or container** with as few privileges as possible, never your own computer.
2. **Keep sensitive data out of reach**, such as logins and personal files.
3. **Limit internet access** to an allowlist of sites.
4. **Have a person confirm anything with real consequences:** purchases, accepting terms, sending messages, deleting data.

**Prompt injection is the special risk.** Claude reads whatever is on screen, and a web page can contain text written to look like instructions. Claude may follow those instead of yours. Anthropic scans screenshots for this, but your sandbox is the real protection.

**To try it**, start from Anthropic's [computer use reference implementation](https://github.com/anthropics/claude-quickstarts/tree/main/computer-use-demo). It packages a container, the tool code, the agent loop and a web page to watch it work. It runs on your API key, so set a spend limit first and give it short tasks: as the cell above showed, every turn is expensive.

## 4. How it all fits together

You've now seen the same ideas from three directions:

| Idea | You built it | Claude Code | Computer use |
|---|---|---|---|
| **The tool loop** | Notebook 05: call, run tools, send results, repeat | Runs it for every task | Runs it with screen actions |
| **Anthropic-defined tools** | Notebook 06: you implement a tool whose schema Claude already knows | Its own built-in tools | The computer toolset: you perform the actions |
| **MCP** | Notebook 10: a server and a client | A built-in MCP client | Not used |
| **Small, general tools** | Notebook 11 | `bash`, read, edit, search, not a special "run tests" tool | Click, type and scroll, not "book a flight" |
| **Seeing the result of each step** | Notebook 11 | Reads files before editing, then runs the tests | A screenshot after each action |

**The main lesson for your own agents:** give Claude a few **general** tools it can combine, and a way to **check the result of every action**. Claude Code and computer use both work because of those two things.

## Takeaways

1. **Claude Code is an agent in your terminal:** the tool loop, with tools for files, shell commands, the web and MCP.
2. **Give it memory with `CLAUDE.md`** (`/init`), and work in three steps: context, plan, then implement.
3. **Plan mode** (`Shift+Tab`, or `--permission-mode plan`) lets Claude explore without editing until you approve.
4. **Claude Code is an MCP client:** `claude mcp add <name> -- <command>` plugs in any MCP server, including your own.
5. **Computer use** is the same loop with a mouse, keyboard and screenshots. On Sonnet 5.5 it's the `computer_toolset_20260801` toolset.
6. **It's expensive and it acts on a real screen:** run it only in an isolated VM, with turn limits and a person approving anything that matters.

## Your turn

These happen outside the notebook. Claude Code and the computer use demo are billed to your account, so keep a spend limit set.

- Run `/init` in this repo with Claude Code and read the `CLAUDE.md` it writes. Does it describe the notebooks correctly?
- Register `halvora-docs` with `claude mcp add`, then check `/mcp` shows it connected.
- **Free:** in section 3, count the tokens with other members disabled too, such as `{"zoom": {"enabled": False}, "triple_click": {"enabled": False}}`. How much does each one save?
- Use plan mode on a small change, such as "add a Your turn exercise to notebook 05". Read the plan, then approve or refine it.

**Next:** [13 · Capstone](13_capstone_board_assistant.ipynb): one assistant that brings together the techniques from the whole repo.